# Gradient Boosting with XGBoost
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/algorithms/xgboost_boosting.ipynb)

XGBoost builds shallow trees sequentially, where each new tree fits the residual errors of the ensemble so far. It dominates tabular-data competitions and usually beats random forests on structured data.

**Covered:** XGBClassifier basics, key hyperparameters, early stopping, comparison against RF.

In [ ]:
!pip install -q xgboost

## 1. Train on the wine dataset

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

wine = load_wine()
Xtr, Xte, ytr, yte = train_test_split(wine.data, wine.target,
                                      test_size=0.25, stratify=wine.target,
                                      random_state=42)

model = XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=4,
                      subsample=0.9, colsample_bytree=0.9,
                      eval_metric="mlogloss", random_state=42)
model.fit(Xtr, ytr)
print("test accuracy:", accuracy_score(yte, model.predict(Xte)))
print(classification_report(yte, model.predict(Xte),
                            target_names=wine.target_names))

### The three knobs that matter most
| Parameter | Meaning | Typical range |
|---|---|---|
| `learning_rate` | step size shrinkage | 0.01 - 0.3 |
| `n_estimators` | number of trees | 100 - 2000 (with early stopping) |
| `max_depth` | tree complexity | 3 - 8 |
`subsample` / `colsample_bytree` < 1 add stochasticity -> regularization.

## 2. Early stopping (anti-overfit)

In [ ]:
model_es = XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=4,
                         eval_metric="mlogloss", early_stopping_rounds=30,
                         random_state=42)
model_es.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
print("best iteration:", model_es.best_iteration)
print("test accuracy :", model_es.score(Xte, yte))

## 3. XGBoost vs RandomForest on the same split

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import pandas as pd
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(Xtr, ytr)
print(f"RandomForest : {rf.score(Xte, yte):.4f}")
print(f"XGBoost      : {model_es.score(Xte, yte):.4f}")

(pd.Series(model.feature_importances_, index=wine.feature_names)
   .nlargest(8).sort_values()
   .plot.barh(title="XGBoost gain-based importance", figsize=(7, 4)))
plt.tight_layout(); plt.show()

**Siblings worth knowing**
- **LightGBM** - faster on large data (histogram splits, leaf-wise growth)
- **CatBoost** - native categorical features, strong defaults

All follow the same gradient-boosting recipe; try them in this order for tabular problems: XGBoost -> LightGBM -> CatBoost.